In [ ]:
import os
from openai import OpenAI
from pinecone import Pinecone
import voyageai
from pinecone_text.sparse import BM25Encoder

openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
vo = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
bm25 = BM25Encoder().load("../data/japanese_bm25_model.json")
index = pc.Index("japanese-wiki-index")


def translate_and_optimize_query(english_query: str) -> str:
    """Translates English anime queries to Japanese, preserving proper nouns

    and character names for optimal database matching.
    """
    system_prompt = (
        "You are an expert translator specializing in Japanese anime, manga, and pop culture. "
        "Your task is to translate user queries from English to Japanese so they can be searched in a Japanese Wikipedia database. "
        "Rules:\n"
        "1. Maintain accurate Japanese titles for shows (e.g., 'Uma musume' -> 'ウマ娘', 'Attack on Titan' -> '進撃の巨人').\n"
        "2. Keep character names accurate in Japanese.\n"
        "3. Return ONLY the final translated search text. Do not provide explanations or extra commentary."
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",  # Highly accurate, extremely cheap, and incredibly fast
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": english_query},
        ],
        temperature=0.0,  # Deterministic output
    )

    japanese_translation = response.choices[0].message.content.strip()

    # Combine the Japanese translation with the original English query keywords
    # This gives Voyage AI and MeCab the best of both worlds to look up.
    optimized_search_string = f"{japanese_translation} {english_query}"

    return optimized_search_string

def query_pinecone(user_english_query: str, alpha: float = 0.35, top_k = 5):
    # Translate the query to Japanese
    search_string = translate_and_optimize_query(user_english_query)
    print(f"Optimized Search String: '{search_string}'")

    # Generate Dense Vector from Voyage (using the translated string)
    dense_response = vo.embed(
        texts=[search_string], model="voyage-4-lite", input_type="query"
    )
    query_dense = dense_response.embeddings[0]

    # Generate Sparse Vector from MeCab BM25
    query_sparse = bm25.encode_queries(search_string)

    # Scale vectors using the Alpha parameter
    scaled_dense = [v * alpha for v in query_dense]
    scaled_sparse = {
        "indices": query_sparse["indices"],
        "values": [v * (1 - alpha) for v in query_sparse["values"]],
    }

    # Query Pinecone
    response = index.query(
        vector=scaled_dense,
        sparse_vector=scaled_sparse,
        top_k=top_k,
        include_metadata=True,
    )
    return response

def answer_question(query, alpha: float = 0.35, top_k = 5):
    results = query_pinecone(query, alpha=alpha, top_k=top_k)
    # Show the results
    print(f"--- Search Results for: '{query}' ---")
    for match in results["matches"]:
        print(f"\n[Score: {match.score:.4f}]")
        print(f"SOURCE: {match.metadata['source']}")
        print(f"MEDIA TYPE: {match.metadata['mediatype']}")
        print("-" * 30)
        print(match.metadata['text'][:400] + "...") # Show first 400 chars

    # Prepare the Context from Pinecone results
    context_list = []
    for match in results["matches"]:
        context_list.append(f"Source: {match.metadata['source']}\nContent: {match.metadata['text']}")

    context_text = "\n\n---\n\n".join(context_list)

    # Create the Prompt
    prompt = f"""
    Answer in English. You are a helpful assistant. Answer the question based ONLY on the context provided below. If the answer isn't in the context, say you don't know.

    Context:
    {context_text}

    Question: {query}
    Answer:"""

    # Generate Answer (Example using OpenAI - requires 'openai' library)
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", # Fast and cheap for RAG
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    print("\n--- FINAL ANSWER ---")
    print(response.choices[0].message.content)

    # =========================================================
    # Run the Test
    # =========================================================

query1 =  "What type of vehicle is Hermes from Kino's Journey"
query2 = "Who is the main character in Uma Musume?"
query3 = "Tell me about Rin Tohsaka"
query4 = "Who are the main characters of Kino's Journey?"
query5 = "Who are the main characters of Bunny Drop or Usagi Drop?"
query6 = "Who is the tank in Kino's Journey?"
query7 = "Who is Kumiko in Hibike Euphonium?"
query8 = "What is the competition in the first season of Hibike Euphonium?"
query9 = "What is the town from Non Non Biyori based on?"
english_user_question = "Who is the fastest horse in Uma musume"
results = answer_question(query9, alpha=0.35)

Optimized Search String: 'のんのんびよりの舞台はどこに基づいていますか？ What is the town from Non Non Biyori based on?'
--- Search Results for: 'What is the town from Non Non Biyori based on?' ---

[Score: 0.2730]
SOURCE: C:\wiki_data_json\manga\のんのんびより.json
MEDIA TYPE: manga
------------------------------
村の人物
加賀山楓（かがやま かえで）
声 - 佐藤利奈
旭丘分校の卒業生で、一穂の後輩。20歳。幼少時よりあだ名は「駄菓子屋」。このみと同じ高校卒で、学校内で会うこともあった。
「かがや」という駄菓子屋を経営しており、丸ポストや、カプセル自販機が置かれ、トンネルの目の前に位置している。
本業の駄菓子屋の他にもスキー用具・布団などのレンタル、通販の取り寄せなどを請け負っている（スキー用具は趣味で持っていた）。しかし、客のほとんどが旭丘分校の生徒であるということもあり、経営状況はあまり良好ではない様子。作中の5年前はおばあちゃんが経営していた。
このみを除く年下の分校生からは本名の楓ではなく、必ず「駄菓子屋（蛍のみ『駄菓子屋さん』）」と呼ばれる。きっかけは、分校入学初日のひかげが「楓姉ちゃん、駄菓子屋に住んでるからあだ名『駄菓子屋』な」と言ったことである。
ボーイッシュな性格で、普段から男言...

[Score: 0.2721]
SOURCE: C:\wiki_data_json\manga\のんのんびより.json
MEDIA TYPE: manga
------------------------------
流行には疎いとは言うものの、小鞠よりははるかに知識が豊富であり、東京育ちの蛍との会話も難なくこなす程度の知識は有している。
たまたま街に行った帰りの電車内で一条一家と乗り合わせ、蛍が家庭内で見せる甘えん坊の姿を知る数少ない人物。ほかにもペチと戯れはしゃいでいる蛍の姿も知っており、いずれの時も困惑していた。
明るく気さくな性格。一方、やや強引で押しが強い他、他人の弱みには鋭く切り込むところがあり、

In [8]:
results

QueryResponse(matches=[ScoredVector(id='238d3012-5baf-5952-9dba-d9023494cc7d', score=0.2059865, values=[], metadata={'author': 'Cygames', 'id': '238d3012-5baf-5952-9dba-d9023494cc7d', 'mediatype': 'anime', 'seq_num': 1, 'source': 'C:\\wiki_data_json\\anime\\うまよん (アニメ).json', 'text': '用語\n共通\nウマ娘\n異世界の競走馬の名前と魂を受け継いで生まれてくる、女性しか存在しない生物。\n基本的には過去にJRAまたは地方競馬に在籍した実在の競走馬の名前が付けられており、誕生日もそれらに準じる。髪および尻尾は多くが実在馬の毛並みと同系色で、顔に星や流星を持つ馬がモデルのウマ娘は、前髪の一部にメッシュが入る。また、メンコやシャドーロールなどの馬具については耳カバーやリボン、マスクといったアクセサリーで表現するなど、一部例外を除いてモデルの実在馬の外見的特徴を受け継いでいる。モデルが牡馬の場合は右耳側に、牝馬の場合は左耳側に、リボンやシュシュなどの何らかの装飾品をつけている 。性格や趣味、他のウマ娘との関係や幼少期のエピソードについても、モデルとなった競走馬（あるいはその主戦騎手）のそれが反映されていることが多い。馬主の職業や特徴が当該ウマ娘の父親または母親に反映されていることもある。\n腰付近から馬のような尻尾が生え、馬のような耳が頭頂部付近にある。耳と尻尾以外は一般的な人間の女性と同様の見た目を有するが、軒並み容姿端麗であるためアイドル的な人気を得ている。\n全力疾走するウマ娘の走行速度はおよそ時速50〜70キロメートルにも達し、公道にウマ娘専用の通行帯が敷かれている場所も存在する。その圧倒的な身体能力を活かしてアスリートとして活動する者が多く、中でもトレセン学園に在籍し、国民的スポーツ・エンターテイメントである「トゥインクル・シリーズ」への参加に向けて特訓に励んでいる、いわゆる「競走ウマ娘」が多数を占める。競走以外の種目においても、身体能力の違いから一般アスリートとは別枠となっている。スポ